## X, y and train-test-split

In [1]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.db import get_engine, fetch_table
from src.config import FEATURE_COLS, FEATURE_COLS_NO_WIFI
from src.config import MODEL_TEXT_COLS, MODEL_NUMBER_COLS, MODEL_RATING_COLS
from src.features import make_features

print("Project root:", project_root)

Project root: d:\OMEN-16\ML-Projects\airline-satisfaction-ml


In [2]:
engine = get_engine()
df = fetch_table(engine)

print("Shape:", df.shape)
print("First id:", df["id"].iloc[0], "| last id:", df["id"].iloc[-1])

Shape: (129880, 24)
First id: 1 | last id: 129880


## Building X

In [3]:
# X holds the columns the model learns from
X = make_features(df)

# the same table without wifi
X_no_wifi = X[FEATURE_COLS_NO_WIFI]

print("X shape        :", X.shape)
print("X_no_wifi shape:", X_no_wifi.shape)

X shape        : (129880, 18)
X_no_wifi shape: (129880, 17)


In [5]:
X.head()

,customer_type,type_of_travel,travel_class,age,flight_distance,departure_delay,ease_of_online_booking,checkin_service,online_boarding,onboard_service,seat_comfort,leg_room_service,cleanliness,food_and_drink,inflight_service,inflight_wifi_service,inflight_entertainment,baggage_handling
0,First-time,Business,Business,48,821,2,3,4,3,3,5,2,5,5,5,3,5,5
1,Returning,Business,Business,35,821,26,2,3,5,5,4,5,5,3,5,2,5,5
2,Returning,Business,Business,41,853,0,4,4,5,3,5,3,5,5,3,4,3,3
3,Returning,Business,Business,50,1905,0,2,3,4,5,5,5,4,4,5,2,5,5
4,Returning,Business,Business,49,3470,0,3,3,5,3,4,4,5,4,3,3,3,3


#### Building y

In [6]:
# y holds the answer we want to predict: 1 = satisfied, 0 = not satisfied
y = (df["satisfaction"] == "Satisfied").astype(int)
y.name = "satisfied"

print("y shape:", y.shape)
print()
print(y.value_counts())
print()
print("Share satisfied:", round(y.mean(), 4))

y shape: (129880,)

satisfied
0    73452
1    56428
Name: count, dtype: int64

Share satisfied: 0.4345


In [7]:
# X and y side by side, first 5 passengers
pd.concat([X.head(), y.head()], axis=1)

,customer_type,type_of_travel,travel_class,age,flight_distance,departure_delay,ease_of_online_booking,checkin_service,online_boarding,onboard_service,seat_comfort,leg_room_service,cleanliness,food_and_drink,inflight_service,inflight_wifi_service,inflight_entertainment,baggage_handling,satisfied
0,First-time,Business,Business,48,821,2,3,4,3,3,5,2,5,5,5,3,5,5,0
1,Returning,Business,Business,35,821,26,2,3,5,5,4,5,5,3,5,2,5,5,1
2,Returning,Business,Business,41,853,0,4,4,5,3,5,3,5,5,3,4,3,3,1
3,Returning,Business,Business,50,1905,0,2,3,4,5,5,5,4,4,5,2,5,5,1
4,Returning,Business,Business,49,3470,0,3,3,5,3,4,4,5,4,3,3,3,3,1


In [8]:
print("Rows in X:", len(X))
print("Rows in y:", len(y))
print("Same number of rows       :", len(X) == len(y))
print("Row numbers line up       :", X.index.equals(y.index))

Rows in X: 129880
Rows in y: 129880
Same number of rows       : True
Row numbers line up       : True


#### checking is anthing in X that shouldn't be

In [9]:
print("'satisfaction' is in X:", "satisfaction" in X.columns)
print("'satisfied' is in X   :", "satisfied" in X.columns)
print("'id' is in X          :", "id" in X.columns)
print()
print("df still has", df.shape[1], "columns and no 'satisfied' column:", "satisfied" not in df.columns)

'satisfaction' is in X: False
'satisfied' is in X   : False
'id' is in X          : False

df still has 24 columns and no 'satisfied' column: True


#### checking blanks and data types

In [10]:
print("Missing values in X:", X.isnull().sum().sum())
print("Missing values in y:", y.isnull().sum())
print()
print(X.dtypes.value_counts())

Missing values in X: 0
Missing values in y: 0

int64    15
str       3
Name: count, dtype: int64


In [11]:
number_cols = MODEL_NUMBER_COLS + MODEL_RATING_COLS

corr = X[number_cols].corrwith(y).abs().sort_values(ascending=False)

print("Three highest correlations with y:")
print(corr.head(3).round(3))

Three highest correlations with y:
online_boarding           0.502
inflight_entertainment    0.398
seat_comfort              0.349
dtype: float64


observation : 
- here the strongest link is 0.50, the same online boarding figure from Step 3. That's what a healthy X looks like. If a column were a copy of y, or gave the answer away, its correlation would be close to 1.0.

- this check can only catch obvious leaks. It's a smoke alarm, not a guarantee. The real protection is rule 4 from 6.5.4: knowing where each column comes from.

In [12]:
# what a leak would look like: a column that is a copy of y
leak_test = X[["age"]].copy()
leak_test["copy_of_y"] = y

print(leak_test.corrwith(y).round(3))

age          0.134
copy_of_y    1.000
dtype: float64


#### what is inside of X

In [13]:
# what a leak would look like: a column that is a copy of y
leak_test = X[["age"]].copy()
leak_test["copy_of_y"] = y

print(leak_test.corrwith(y).round(3))

age          0.134
copy_of_y    1.000
dtype: float64


In [14]:
for col in MODEL_TEXT_COLS:
    print(col, "->", sorted(X[col].unique()))

customer_type -> ['First-time', 'Returning']
type_of_travel -> ['Business', 'Personal']
travel_class -> ['Business', 'Economy', 'Economy Plus']


In [15]:
X[MODEL_NUMBER_COLS + MODEL_RATING_COLS].describe().T[["count", "mean", "min", "max"]].round(2)

,count,mean,min,max
age,129880.0,39.43,7.0,85.0
flight_distance,129880.0,1190.32,31.0,4983.0
departure_delay,129880.0,14.71,0.0,1592.0
ease_of_online_booking,129880.0,2.76,0.0,5.0
checkin_service,129880.0,3.31,0.0,5.0
online_boarding,129880.0,3.25,0.0,5.0
onboard_service,129880.0,3.38,0.0,5.0
seat_comfort,129880.0,3.44,0.0,5.0
leg_room_service,129880.0,3.35,0.0,5.0
cleanliness,129880.0,3.29,0.0,5.0


#### The one column that seperates column X from X_no_wifi

In [16]:
print("Only in X         :", set(X.columns) - set(X_no_wifi.columns))
print("Only in X_no_wifi :", set(X_no_wifi.columns) - set(X.columns))

Only in X         : {'inflight_wifi_service'}
Only in X_no_wifi : set()
